In [11]:
pip install -r requirements.txt


  Using cached openai-2.44.0-py3-none-any.whl.metadata (34 kB)
  Using cached langchain-1.3.11-py3-none-any.whl.metadata (6.0 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_openai-1.3.3-py3-none-any.whl.metadata (3.4 kB)
  Using cached langgraph-1.2.7-py3-none-any.whl.metadata (4.9 kB)
  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached notebook-7.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached tiktoken-0.13.0-cp312-cp312-win_amd64.whl.metadata (6.8 kB)
  Using cached faiss_cpu-1.14.3-cp312-cp312-win_amd64.whl.metadata (7.8 kB)
  Using cached anyio-4.14.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jiter-0.16.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached idna-3.18-py3-none-any.whl.met


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from __future__ import annotations

import operator
from typing import TypedDict, List, Annotated

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage


In [13]:
class Task(BaseModel):
    id: int
    title: str
    brief: str = Field(..., description="What to cover")

In [ ]:
class Plan(BaseModel):
    blog_title: str
    tasks: List[Task]

In [16]:
class State(TypedDict):
    topic: str
    plan: Plan
    # reducer: results from workers get concatenated automatically
    final: str

In [ ]:
llm = ChatOpenAI(model="gpt-4.1-mini")

In [ ]:
def orchestrator(state: State) -> dict:
    plan = llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(
                content=(
                    "Create a blog plan with 5-7 sections on the following topic."
                )
            ), 
            HumanMessage(content=f"Topic: {state['topic']} ")

        ]
    )

    return {"plan": plan}

In [ ]:
def fanout(state: State):
    return [Send("worker", {"task": task, "topic": state["topic"], "plan": state["plan"]})
            for task in state["plan"].tasks
            ]

In [ ]:
def worker(payload: dict) -> dict:

    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]

    blog_title = plan.blog_title
    
    section_md = llm.invoke(
        [
        SystemMessage(content="Write one clean Markdown section."),
        HumanMessage(content =(
            f"Blog: {blog_title}\n",
            f"Topic: {topic}\n\n"
            f"Section: {task.title}\n"
            f"Brief: {task.brief}\n\n"
            "Return only the section content in Markdown."
                )   
            ),
        ]
    ).content.strip()

    return {"sections": [section_md]}


In [ ]:

g =  StateGraph(State)
g.add_node("orchestrator", orchestrator)
g.add_node("worker", worker)
g.add_node("reducer", reducer)

In [ ]:
g.add_edge(START, "orchestrator")
g.add_edge("orchestrator", "worker")
g.add_edge("worker", "reducer")
g.add_edge("reducer", END)

app = g.compile()

app

In [ ]:
out = app.invoke({"topic": "Write a blog on Self Attention", "sections": []})